# Step 40 — Build hypocenter indexes (former steps 40 and 41)

Ingest two complementary sources without merging their records:

1. `Pinatubo_all.sum`: fixed-column HYPO71 summary.
2. PINAALL: choose original `PINAALL.DAT` or the annotated Randy White catalog.

Each source has separate settings, run and review cells. Run either section independently after the common setup, or run both. All three parser implementations remain separate and unchanged. These cells write indexes and error logs; comparison and association remain in steps 42–44.

Canonical original-source output filenames are unchanged. Annotated PINAALL outputs retain their separate names; explicitly select that index in downstream notebooks if you use it. Existing outputs follow the parsers' overwrite behavior. Review the source paths before running.

The summary section now uses the authoritative VDAP HYPO71 reader through a CSV-schema adapter. Arithmetic clock rollover is corrected and blank quality fields are allowed. PINAALL parsers are unchanged.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
from pinatubo_steps.step_40_build_hypocenter_index_pinatubo_all import run as run_summary
from pinatubo_steps.step_41_build_hypocenter_index_pinaall import run as run_original
from pinatubo_steps.step_41_build_hypocenter_index_pinaall_v2 import run as run_annotated

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

def preview_result(result):
    for name, value in result.items():
        print(name)
        display(value.head() if hasattr(value, 'head') else
                value[:5] if isinstance(value, list) else value)


## HYPO71 summary — Pinatubo_all.sum

This produces `40_summary_file_index.csv` and the unparsed-line log. It does not depend on the PINAALL section below.


In [ ]:
SUMMARY_SETTINGS = dict(
    summary_file=P['SUMMARY_FILE_40'],
    out_csv=P['SUMMARY_FILE_40_INDEX'],
    error_log=P['SUMMARY_FILE_40_ERR'],
)
display(SUMMARY_SETTINGS)


In [ ]:
summary_result = run_summary(**SUMMARY_SETTINGS)


In [ ]:
print('Summary index:', SUMMARY_SETTINGS['out_csv'])
preview_result(summary_result)


## PINAALL — original or annotated parser

Choose `PARSER = "original"` (default) or `"annotated"`, then rerun this section's settings cell. The original parser writes `41_summary_file_index.csv`. The annotated parser writes `41_annotated_summary_file_index.csv`, plus annotation, error and audit files.

`LOCAL_UTC_OFFSET = None` leaves unresolved local-time records unresolved. Use `8.0` only when adopting the documented Philippine UTC+08 conversion. Inspect annotated time-basis and transition-date warnings before UTC association. The second numeric field is retained as magnitude for compatibility, but its interpretation needs confirmation.

This section can run independently of the summary-file section. Selecting annotated input does not replace the canonical step-41 index used downstream.


In [ ]:
PARSER = 'original'  # Choose 'original' or 'annotated'.
LOCAL_UTC_OFFSET = None  # Annotated parser only; 8.0 means Philippine UTC+08.

if PARSER == 'original':
    run_pinaall = run_original
    PINAALL_SETTINGS = dict(
        pinaall_file=P['SUMMARY_FILE_41'],
        out_csv=P['SUMMARY_FILE_41_INDEX'],
        error_log=P['SUMMARY_FILE_41_ERR'],
    )
elif PARSER == 'annotated':
    run_pinaall = run_annotated
    PINAALL_SETTINGS = dict(
        source=str(Path(P['LEGACY_HYPO_DIR']) / 'PINAALL_annotated_RW_20261013.DAT'),
        out_csv=str(Path(P['TMP_DIR']) / '41_annotated_summary_file_index.csv'),
        annotations_csv=str(Path(P['TMP_DIR']) / '41_annotations.csv'),
        error_log=str(Path(P['TMP_DIR']) / '41_annotated_parse_errors.csv'),
        audit_report=str(Path(P['TMP_DIR']) / '41_annotated_parse_summary.txt'),
        local_utc_offset=LOCAL_UTC_OFFSET,
    )
else:
    raise ValueError("PARSER must be 'original' or 'annotated'")

print('Selected PINAALL parser:', PARSER)
display(PINAALL_SETTINGS)


In [ ]:
pinaall_result = run_pinaall(**PINAALL_SETTINGS)


In [ ]:
print('PINAALL parser:', PARSER)
print('PINAALL index:', PINAALL_SETTINGS['out_csv'])
preview_result(pinaall_result)


## Continue with comparison and association

Steps 42–44 still consume the original `40_summary_file_index.csv` and `41_summary_file_index.csv` paths by default. If you chose annotated PINAALL, set their `hypo41` input to `PINAALL_SETTINGS['out_csv']` after reviewing time-basis flags.

The source records remain in separate tables; step 43 performs the later association. Review both parsers' error logs before continuing. Former source-specific notebooks are preserved under `superseded/hypocenter_ingestion`.
